In [ ]:
########################################################################
# Inclusão das Bibliotecas Necessárias
########################################################################
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
########################################################################
# Localizando o Diretório Base
########################################################################
%cd /content/drive/MyDrive/Atividades/PUC-DI/LocalMultiAgente/Código



/content/drive/MyDrive/Atividades/PUC-DI/LocalMultiAgente/Código


In [1]:
"""
VERSÃO QMIX COM FALHAS ALEATÓRIAS DOS ROBÔS - SEM OBSTÁCULOS Y
- Todas as células 'Y' foram substituídas por '0' (espaço vazio)
- 20% de chance de falha no movimento
- Robôs podem escolher direções alternativas quando falham
- Armazena gráficos individuais e logs detalhados
"""

import numpy as np
import gymnasium as gym
from gymnasium import spaces
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from collections import deque
import random
from typing import List, Tuple, Dict, Optional
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from datetime import datetime
import json
import os
from pathlib import Path
import warnings
import time
import gc

warnings.filterwarnings('ignore')

# ==================== CONFIGURAÇÃO ====================
MAP_CONFIG = {
    'height': 12,
    'width': 8,
    'grid': [
        ['R1', '0', '0', '0', '0', '0', '0', 'R2'],
        ['0', '0', 'A', 'A', 'A', 'A', '0', '0'],
        ['0', '0', 'A', 'A', 'A', 'A', '0', '0'],
        ['X', '0', '0', '0', 'X', '0', '0', '0'],
        ['0', '0', 'X', '0', '0', '0', '0', '0'],
        ['0', '0', '0', 'X', '0', '0', 'X', 'X'],
        ['0', 'X', '0', '0', '0', 'X', '0', '0'],
        ['0', '0', '0', 'X', '0', '0', '0', '0'],
        ['X', '0', '0', '0', '0', '0', 'X', '0'],
        ['X', '0', 'B', 'B', 'B', 'B', 'X', '0'],
        ['X', '0', 'B', 'B', 'B', 'B', '0', '0'],
        ['0', '0', '0', '0', '0', '0', '0', '0']
    ]
}

class Config:
    # Exploração
    EPSILON_START = 1.0
    EPSILON_END = 0.05
    EPSILON_DECAY_STEPS = 50000

    # Aprendizado
    LEARNING_RATE = 0.0001
    BATCH_SIZE = 128
    GAMMA = 0.95
    TAU = 0.001
    WEIGHT_DECAY = 1e-5

    # Rede Neural
    HIDDEN_DIM = 256
    DROPOUT_RATE = 0.2
    MIXER_HIDDEN_DIM = 128

    # Memória
    BUFFER_SIZE = 100000
    PRIORITIZED_REPLAY = True
    ALPHA = 0.6
    BETA = 0.4

    # Treinamento
    MAX_GRAD_NORM = 1.0
    LEARNING_STARTS = 1000
    TRAIN_FREQ = 4
    USE_SOFT_UPDATE = True
    MAX_STEPS = 500
    EPISODES_PER_SESSION = 1500

    # Sistema
    SAVE_INTERVAL = 100
    CLEAN_MEMORY_EVERY = 500

    # Configurações de falha
    FAILURE_PROBABILITY = 0.2
    FAILURE_PENALTY = -0.15
    ALTERNATIVE_DIRECTIONS = True
    LOG_FAILURES = False  # Desativado para não mostrar mensagens


# ==================== PRIORITIZED REPLAY BUFFER PARA QMIX ====================
class QMIXPrioritizedReplayBuffer:
    def __init__(self, capacity, alpha=0.6, beta=0.4):
        self.capacity = capacity
        self.alpha = alpha
        self.beta = beta
        self.buffer = []
        self.priorities = []
        self.position = 0

    def push(self, state, actions, rewards, next_state, done, global_state, next_global_state):
        max_priority = max(self.priorities) if self.priorities else 1.0

        if len(self.buffer) < self.capacity:
            self.buffer.append((state, actions, rewards, next_state, done, global_state, next_global_state))
            self.priorities.append(max_priority)
        else:
            self.buffer[self.position] = (state, actions, rewards, next_state, done, global_state, next_global_state)
            self.priorities[self.position] = max_priority

        self.position = (self.position + 1) % self.capacity

    def sample(self, batch_size):
        if len(self.buffer) == self.capacity:
            priorities = np.array(self.priorities)
        else:
            priorities = np.array(self.priorities[:len(self.buffer)])

        probs = priorities ** self.alpha
        probs /= probs.sum()

        indices = np.random.choice(len(self.buffer), batch_size, p=probs)

        total = len(self.buffer)
        weights = (total * probs[indices]) ** (-self.beta)
        weights /= weights.max()

        batch = [self.buffer[idx] for idx in indices]
        states, actions, rewards, next_states, dones, global_states, next_global_states = zip(*batch)

        return (np.array(states), np.array(actions), np.array(rewards),
                np.array(next_states), np.array(dones), np.array(global_states),
                np.array(next_global_states), indices, weights)

    def update_priorities(self, indices, td_errors):
        for idx, td_error in zip(indices, td_errors):
            self.priorities[idx] = abs(td_error) + 1e-6

    def __len__(self):
        return len(self.buffer)


# ==================== REDES NEURAIS QMIX ====================
class DQN(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_dim=256, dropout=0.2):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight)
            nn.init.constant_(module.bias, 0.0)

    def forward(self, x):
        return self.network(x)


class QMixer(nn.Module):
    def __init__(self, n_agents, state_dim, hidden_dim=128):
        super().__init__()

        self.n_agents = n_agents
        self.state_dim = state_dim
        self.hidden_dim = hidden_dim

        self.hyper_w1 = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim * n_agents)
        )

        self.hyper_w2 = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

        self.hyper_b1 = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

        self.hyper_b2 = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, agent_qs, states):
        batch_size = agent_qs.size(0)

        w1 = self.hyper_w1(states).view(batch_size, self.hidden_dim, self.n_agents)
        b1 = self.hyper_b1(states).view(batch_size, self.hidden_dim, 1)

        hidden = torch.bmm(w1, agent_qs.unsqueeze(2)) + b1
        hidden = torch.relu(hidden)

        w2 = self.hyper_w2(states).view(batch_size, 1, self.hidden_dim)
        b2 = self.hyper_b2(states).view(batch_size, 1, 1)

        q_total = torch.bmm(w2, hidden) + b2

        return q_total.squeeze(-1)


# ==================== AGENTE QMIX ====================
class QMIXAgent:
    def __init__(self, agent_id, state_dim, action_dim, config, n_agents, global_state_dim):
        self.agent_id = agent_id
        self.action_dim = action_dim
        self.config = config
        self.n_agents = n_agents
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        self.failure_count = 0
        self.successful_moves = 0

        self.policy_net = DQN(state_dim, action_dim, config.HIDDEN_DIM, config.DROPOUT_RATE).to(self.device)
        self.target_net = DQN(state_dim, action_dim, config.HIDDEN_DIM, config.DROPOUT_RATE).to(self.device)
        self.target_net.load_state_dict(self.policy_net.state_dict())

        self.optimizer = optim.Adam(self.policy_net.parameters(),
                                   lr=config.LEARNING_RATE,
                                   weight_decay=config.WEIGHT_DECAY)

        self.steps_done = 0
        self.learning_steps = 0
        self.total_episodes = 0
        self.losses = []

    def get_epsilon(self):
        if self.steps_done >= self.config.EPSILON_DECAY_STEPS:
            return self.config.EPSILON_END

        epsilon = self.config.EPSILON_START - (self.config.EPSILON_START - self.config.EPSILON_END) * \
                  self.steps_done / self.config.EPSILON_DECAY_STEPS
        return max(self.config.EPSILON_END, epsilon)

    def select_action(self, state, training=True):
        self.steps_done += 1

        if training and random.random() < self.get_epsilon():
            return random.randrange(self.action_dim)

        with torch.no_grad():
            state_tensor = torch.FloatTensor(state).unsqueeze(0).to(self.device)
            q_values = self.policy_net(state_tensor)
            return q_values.argmax().item()

    def get_q_values(self, states):
        if isinstance(states, np.ndarray):
            states_tensor = torch.FloatTensor(states).to(self.device)
        else:
            states_tensor = states.to(self.device)
        return self.policy_net(states_tensor)

    def get_target_q_values(self, states):
        if isinstance(states, np.ndarray):
            states_tensor = torch.FloatTensor(states).to(self.device)
        else:
            states_tensor = states.to(self.device)
        return self.target_net(states_tensor)

    def soft_update_target(self):
        for target_param, policy_param in zip(self.target_net.parameters(),
                                               self.policy_net.parameters()):
            target_param.data.copy_(self.config.TAU * policy_param.data +
                                   (1 - self.config.TAU) * target_param.data)

    def reset_failure_stats(self):
        self.failure_count = 0
        self.successful_moves = 0


# ==================== AMBIENTE WAREHOUSE COM LOG DE MOVIMENTOS ====================
class WarehouseEnv(gym.Env):
    metadata = {'render.modes': ['rgb_array']}

    def __init__(self, config=None):
        super().__init__()

        self.config = config or Config()
        self.height = MAP_CONFIG['height']
        self.width = MAP_CONFIG['width']
        self.grid = [row[:] for row in MAP_CONFIG['grid']]

        self.robot_positions = None
        self.box_positions = None
        self.targets = self._find_positions('B')

        self.num_robots = 2
        self.num_boxes = len(self._find_positions('A'))
        self.num_targets = len(self.targets)

        self.delivered_boxes = None
        self.steps = 0
        self.max_steps = self.config.MAX_STEPS
        self.total_deliveries = 0
        self.collisions = 0
        self.failures = [0, 0]
        self.distance_traveled = [0, 0]
        self.previous_distances = None

        # Log de movimentos
        self.movement_log = []

        self.action_space = spaces.Tuple([spaces.Discrete(6) for _ in range(self.num_robots)])

        obs_dim = (self.num_robots * 2) + (self.num_boxes * 2) + (self.num_targets * 2) + 8
        self.observation_space = spaces.Box(
            low=-1, high=self.height + self.width,
            shape=(obs_dim,),
            dtype=np.float32
        )

        self.frame_buffer = []

    def _find_positions(self, symbols):
        if isinstance(symbols, str):
            symbols = [symbols]
        positions = []
        for i in range(self.height):
            for j in range(self.width):
                cell = self.grid[i][j]
                if any(cell.startswith(sym) for sym in symbols):
                    positions.append((i, j))
        return positions

    def reset(self):
        self.grid = [row[:] for row in MAP_CONFIG['grid']]
        self.robot_positions = self._find_positions('R')
        self.box_positions = self._find_positions('A')
        self.delivered_boxes = [False] * self.num_boxes

        self.steps = 0
        self.total_deliveries = 0
        self.collisions = 0
        self.failures = [0, 0]
        self.distance_traveled = [0, 0]
        self.movement_log = []

        self.previous_distances = [self._min_distance_to_boxes(r) for r in range(self.num_robots)]

        return self._get_observation(), self._get_info()

    def _min_distance_to_boxes(self, robot_id):
        robot_pos = self.robot_positions[robot_id]
        remaining_boxes = [box_pos for box_pos in self.box_positions
                          if box_pos is not None and
                          not self.delivered_boxes[self.box_positions.index(box_pos)]]

        if not remaining_boxes:
            return 0
        return min([abs(robot_pos[0] - box_pos[0]) + abs(robot_pos[1] - box_pos[1])
                   for box_pos in remaining_boxes])

    def _is_valid_position(self, pos, robot_id=None):
        i, j = pos
        if i < 0 or i >= self.height or j < 0 or j >= self.width:
            return False

        cell = self.grid[i][j]
        if cell == 'X':
            return False

        if robot_id is not None:
            for rid, rpos in enumerate(self.robot_positions):
                if rid != robot_id and rpos == (i, j):
                    return False
        return True

    def _get_direction_name(self, action):
        """Retorna o nome da direção baseado na ação"""
        direcoes = {0: "CIMA", 1: "BAIXO", 2: "ESQUERDA", 3: "DIREITA", 4: "PEGAR", 5: "SOLTAR", 6: "PARADO"}
        return direcoes.get(action, "DESCONHECIDO")

    def _get_alternative_direction(self, original_action, robot_id):
        i, j = self.robot_positions[robot_id]
        alternative_actions = [a for a in range(4) if a != original_action]
        random.shuffle(alternative_actions)

        for alt_action in alternative_actions:
            alt_i, alt_j = i, j
            if alt_action == 0: alt_i -= 1
            elif alt_action == 1: alt_i += 1
            elif alt_action == 2: alt_j -= 1
            elif alt_action == 3: alt_j += 1

            if self._is_valid_position((alt_i, alt_j), robot_id):
                return alt_action, (alt_i, alt_j)

        return None, None

    def _move_robot_with_failure(self, robot_id, action):
        i, j = self.robot_positions[robot_id]
        new_i, new_j = i, j
        movimento_original = self._get_direction_name(action)

        if action == 0: new_i -= 1
        elif action == 1: new_i += 1
        elif action == 2: new_j -= 1
        elif action == 3: new_j += 1

        desired_valid = self._is_valid_position((new_i, new_j), robot_id)
        movimento_final = movimento_original
        pos_final = (new_i, new_j)
        falha_ocorreu = False

        if random.random() < self.config.FAILURE_PROBABILITY:
            self.failures[robot_id] += 1
            falha_ocorreu = True

            if self.config.ALTERNATIVE_DIRECTIONS:
                alt_action, alt_pos = self._get_alternative_direction(action, robot_id)
                if alt_action is not None:
                    movimento_final = self._get_direction_name(alt_action)
                    pos_final = alt_pos

                    old_pos = self.robot_positions[robot_id]
                    self.grid[old_pos[0]][old_pos[1]] = '0'
                    self.grid[alt_pos[0]][alt_pos[1]] = f'R{robot_id + 1}'
                    self.robot_positions[robot_id] = alt_pos
                    self.distance_traveled[robot_id] += 1

                    # Registrar movimento no log
                    self.movement_log.append({
                        'robo': f'R{robot_id+1}',
                        'pos_original_x': i,
                        'pos_original_y': j,
                        'pos_final_x': alt_pos[0],
                        'pos_final_y': alt_pos[1],
                        'movimento_original': movimento_original,
                        'movimento_final': movimento_final,
                        'falha': falha_ocorreu
                    })

                    return self.config.FAILURE_PENALTY - 0.01

            # Se não conseguiu se mover alternativamente
            self.movement_log.append({
                'robo': f'R{robot_id+1}',
                'pos_original_x': i,
                'pos_original_y': j,
                'pos_final_x': i,
                'pos_final_y': j,
                'movimento_original': movimento_original,
                'movimento_final': "PARADO",
                'falha': falha_ocorreu
            })
            return self.config.FAILURE_PENALTY

        # Movimento normal (sem falha)
        if desired_valid:
            self.distance_traveled[robot_id] += 1
            old_pos = self.robot_positions[robot_id]
            self.grid[old_pos[0]][old_pos[1]] = '0'
            self.grid[new_i][new_j] = f'R{robot_id + 1}'
            self.robot_positions[robot_id] = (new_i, new_j)

            self.movement_log.append({
                'robo': f'R{robot_id+1}',
                'pos_original_x': i,
                'pos_original_y': j,
                'pos_final_x': new_i,
                'pos_final_y': new_j,
                'movimento_original': movimento_original,
                'movimento_final': movimento_original,
                'falha': falha_ocorreu
            })
            return -0.005
        else:
            self.collisions += 1
            self.movement_log.append({
                'robo': f'R{robot_id+1}',
                'pos_original_x': i,
                'pos_original_y': j,
                'pos_final_x': i,
                'pos_final_y': j,
                'movimento_original': movimento_original,
                'movimento_final': "BLOQUEADO",
                'falha': falha_ocorreu
            })
            return -0.02

    def _pickup_box(self, robot_id):
        robot_pos = self.robot_positions[robot_id]
        for box_id, box_pos in enumerate(self.box_positions):
            if not self.delivered_boxes[box_id] and box_pos == robot_pos:
                self.box_positions[box_id] = None
                self.grid[robot_pos[0]][robot_pos[1]] = f'R{robot_id + 1}'

                self.movement_log.append({
                    'robo': f'R{robot_id+1}',
                    'pos_original_x': robot_pos[0],
                    'pos_original_y': robot_pos[1],
                    'pos_final_x': robot_pos[0],
                    'pos_final_y': robot_pos[1],
                    'movimento_original': "PEGAR",
                    'movimento_final': "PEGOU",
                    'falha': False
                })
                return 2.0
        return -0.02

    def _drop_box(self, robot_id):
        robot_pos = self.robot_positions[robot_id]

        box_with_robot = None
        for box_id, box_pos in enumerate(self.box_positions):
            if box_pos is None and not self.delivered_boxes[box_id]:
                box_with_robot = box_id
                break

        if box_with_robot is None:
            return -0.02

        for target_pos in self.targets:
            if robot_pos == target_pos:
                self.delivered_boxes[box_with_robot] = True
                self.total_deliveries += 1
                self.grid[robot_pos[0]][robot_pos[1]] = f'R{robot_id + 1}'

                self.movement_log.append({
                    'robo': f'R{robot_id+1}',
                    'pos_original_x': robot_pos[0],
                    'pos_original_y': robot_pos[1],
                    'pos_final_x': robot_pos[0],
                    'pos_final_y': robot_pos[1],
                    'movimento_original': "SOLTAR",
                    'movimento_final': "ENTREGOU",
                    'falha': False
                })
                return 25.0

        return -0.05

    def _calculate_shaped_reward(self, robot_id, base_reward):
        reward = base_reward

        current_distance = self._min_distance_to_boxes(robot_id)
        previous_distance = self.previous_distances[robot_id]

        if current_distance < previous_distance:
            reward += 0.1 * (previous_distance - current_distance)
        elif current_distance > previous_distance:
            reward -= 0.02 * (current_distance - previous_distance)

        self.previous_distances[robot_id] = current_distance

        if all(self.delivered_boxes):
            reward += 50.0

        return reward

    def _get_observation(self):
        obs = []

        for robot_pos in self.robot_positions:
            obs.append(robot_pos[0] / self.height)
            obs.append(robot_pos[1] / self.width)

        for box_id, box_pos in enumerate(self.box_positions):
            if box_pos is None or self.delivered_boxes[box_id]:
                obs.append(-1)
                obs.append(-1)
            else:
                obs.append(box_pos[0] / self.height)
                obs.append(box_pos[1] / self.width)

        for target_pos in self.targets:
            obs.append(target_pos[0] / self.height)
            obs.append(target_pos[1] / self.width)

        for robot_pos in self.robot_positions:
            min_box_dist = min([abs(robot_pos[0] - box_pos[0]) + abs(robot_pos[1] - box_pos[1])
                               for box_pos in self.box_positions
                               if box_pos is not None and
                               not self.delivered_boxes[self.box_positions.index(box_pos)]],
                              default=100)
            obs.append(min_box_dist / (self.height + self.width))

            min_target_dist = min([abs(robot_pos[0] - target_pos[0]) + abs(robot_pos[1] - target_pos[1])
                                  for target_pos in self.targets],
                                 default=100)
            obs.append(min_target_dist / (self.height + self.width))

        return np.array(obs, dtype=np.float32)

    def get_global_state(self):
        global_state = []

        for robot_pos in self.robot_positions:
            global_state.append(robot_pos[0] / self.height)
            global_state.append(robot_pos[1] / self.width)

        for box_pos in self.box_positions:
            if box_pos is not None:
                global_state.append(box_pos[0] / self.height)
                global_state.append(box_pos[1] / self.width)
            else:
                global_state.append(-1)
                global_state.append(-1)

        for delivered in self.delivered_boxes:
            global_state.append(1.0 if delivered else 0.0)

        return np.array(global_state, dtype=np.float32)

    def step(self, actions):
        self.steps += 1

        if len(actions) != self.num_robots:
            actions = [actions] * self.num_robots

        total_reward = 0
        rewards = [0, 0]

        movement_rewards = []
        for robot_id, action in enumerate(actions):
            if action < 4:
                reward = self._move_robot_with_failure(robot_id, action)
                movement_rewards.append(reward)
            else:
                movement_rewards.append(0)

        interaction_rewards = []
        for robot_id, action in enumerate(actions):
            if action == 4:
                reward = self._pickup_box(robot_id)
                interaction_rewards.append(reward)
            elif action == 5:
                reward = self._drop_box(robot_id)
                interaction_rewards.append(reward)
            else:
                interaction_rewards.append(0)

        for robot_id in range(self.num_robots):
            base_reward = movement_rewards[robot_id] + interaction_rewards[robot_id]
            shaped_reward = self._calculate_shaped_reward(robot_id, base_reward)
            rewards[robot_id] = shaped_reward
            total_reward += shaped_reward

        terminated = all(self.delivered_boxes)
        truncated = self.steps >= self.max_steps

        observation = self._get_observation()
        info = self._get_info()

        return observation, rewards, terminated, truncated, info

    def _get_info(self):
        return {
            'steps': self.steps,
            'total_deliveries': self.total_deliveries,
            'collisions': self.collisions,
            'failures_r1': self.failures[0],
            'failures_r2': self.failures[1],
            'distance_traveled': self.distance_traveled.copy(),
            'remaining_boxes': sum(1 for d in self.delivered_boxes if not d),
            'success_rate': self.total_deliveries / self.num_boxes if self.steps > 0 else 0,
        }

    def get_movement_log(self):
        return self.movement_log

    def close(self):
        pass


# ==================== QMIX TRAINER ====================
class QMIXTrainer:
    def __init__(self, agents, mixer, config, state_dim, global_state_dim):
        self.agents = agents
        self.mixer = mixer
        self.config = config
        self.state_dim = state_dim
        self.global_state_dim = global_state_dim
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        self.memory = QMIXPrioritizedReplayBuffer(config.BUFFER_SIZE, alpha=config.ALPHA)
        self.mixer_optimizer = optim.Adam(mixer.parameters(), lr=config.LEARNING_RATE)
        self.learning_steps = 0

    def remember(self, state, actions, rewards, next_state, done, global_state, next_global_state):
        self.memory.push(state, actions, rewards, next_state, done, global_state, next_global_state)

    def optimize(self):
        if len(self.memory) < self.config.BATCH_SIZE:
            return 0

        self.learning_steps += 1

        if self.learning_steps % self.config.TRAIN_FREQ != 0:
            return 0

        (states, actions, rewards, next_states, dones,
         global_states, next_global_states, indices, weights) = self.memory.sample(self.config.BATCH_SIZE)

        states = torch.FloatTensor(states).to(self.device)
        actions = torch.LongTensor(actions).to(self.device)
        rewards = torch.FloatTensor(rewards).to(self.device)
        next_states = torch.FloatTensor(next_states).to(self.device)
        dones = torch.FloatTensor(dones).to(self.device)
        global_states = torch.FloatTensor(global_states).to(self.device)
        next_global_states = torch.FloatTensor(next_global_states).to(self.device)
        weights = torch.FloatTensor(weights).to(self.device)

        curr_qs = []
        for i, agent in enumerate(self.agents):
            agent_qs = agent.get_q_values(states)
            curr_qs.append(agent_qs.gather(1, actions[:, i].unsqueeze(1)))

        curr_qs = torch.cat(curr_qs, dim=1)

        target_qs = []
        for i, agent in enumerate(self.agents):
            agent_target_qs = agent.get_target_q_values(next_states)
            target_qs.append(agent_target_qs.max(1, keepdim=True)[0])

        target_qs = torch.cat(target_qs, dim=1)

        curr_q_total = self.mixer(curr_qs, global_states)

        with torch.no_grad():
            target_q_total = self.mixer(target_qs, next_global_states)
            target = rewards.sum(dim=1, keepdim=True) + self.config.GAMMA * target_q_total * (1 - dones.unsqueeze(1))

        td_errors = (target - curr_q_total).squeeze()
        loss = (weights * td_errors.pow(2)).mean()

        self.mixer_optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.mixer.parameters(), self.config.MAX_GRAD_NORM)
        self.mixer_optimizer.step()

        for agent in self.agents:
            agent_optimizer = agent.optimizer
            agent_optimizer.zero_grad()

            agent_qs = agent.get_q_values(states)
            agent_curr_qs = agent_qs.gather(1, actions[:, agent.agent_id].unsqueeze(1))

            agent_loss = (weights * (target - curr_q_total).detach() * agent_curr_qs).mean()
            agent_loss.backward()
            torch.nn.utils.clip_grad_norm_(agent.policy_net.parameters(), self.config.MAX_GRAD_NORM)
            agent_optimizer.step()

            agent.soft_update_target()

        priorities = td_errors.abs().detach().cpu().numpy() + 1e-6
        self.memory.update_priorities(indices, priorities)

        return loss.item()


# ==================== FUNÇÕES DE PLOTAGEM ====================
def plot_individual_graphs(metrics, save_dir):
    """Plota e salva gráficos individuais"""

    episodes = range(1, len(metrics['episode_rewards']) + 1)

    # 1. Recompensa por episódio
    plt.figure(figsize=(12, 6))
    plt.plot(episodes, metrics['episode_rewards'], 'b-', alpha=0.5, linewidth=1)
    if len(metrics['episode_rewards']) >= 100:
        moving_avg = np.convolve(metrics['episode_rewards'], np.ones(100)/100, mode='valid')
        plt.plot(range(100, len(metrics['episode_rewards']) + 1), moving_avg, 'r-', linewidth=2, label='Média Móvel (100)')
    plt.xlabel('Episódio')
    plt.ylabel('Recompensa')
    plt.title('Recompensa por Episódio')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig(save_dir / 'grafico_recompensa.png', dpi=150, bbox_inches='tight')
    plt.close()

    # 2. Entregas por episódio
    plt.figure(figsize=(12, 6))
    plt.plot(episodes, metrics['episode_deliveries'], 'g-', alpha=0.5, linewidth=1)
    if len(metrics['episode_deliveries']) >= 100:
        moving_avg = np.convolve(metrics['episode_deliveries'], np.ones(100)/100, mode='valid')
        plt.plot(range(100, len(metrics['episode_deliveries']) + 1), moving_avg, 'r-', linewidth=2, label='Média Móvel (100)')
    plt.axhline(y=8, color='orange', linestyle='--', label='Meta (8 caixas)')
    plt.xlabel('Episódio')
    plt.ylabel('Entregas')
    plt.title('Entregas por Episódio')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig(save_dir / 'grafico_entregas.png', dpi=150, bbox_inches='tight')
    plt.close()

    # 3. Steps por episódio
    plt.figure(figsize=(12, 6))
    plt.plot(episodes, metrics['episode_steps'], 'orange', alpha=0.5, linewidth=1)
    if len(metrics['episode_steps']) >= 100:
        moving_avg = np.convolve(metrics['episode_steps'], np.ones(100)/100, mode='valid')
        plt.plot(range(100, len(metrics['episode_steps']) + 1), moving_avg, 'r-', linewidth=2, label='Média Móvel (100)')
    plt.xlabel('Episódio')
    plt.ylabel('Steps')
    plt.title('Steps por Episódio')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig(save_dir / 'grafico_steps.png', dpi=150, bbox_inches='tight')
    plt.close()

    # 4. Taxa de Sucesso por episódio
    plt.figure(figsize=(12, 6))
    plt.plot(episodes, metrics['success_rates'], 'purple', alpha=0.5, linewidth=1)
    if len(metrics['success_rates']) >= 100:
        moving_avg = np.convolve(metrics['success_rates'], np.ones(100)/100, mode='valid')
        plt.plot(range(100, len(metrics['success_rates']) + 1), moving_avg, 'r-', linewidth=2, label='Média Móvel (100)')
    plt.axhline(y=0.95, color='green', linestyle='--', label='Meta 95%')
    plt.xlabel('Episódio')
    plt.ylabel('Taxa de Sucesso')
    plt.title('Taxa de Sucesso por Episódio')
    plt.ylim([0, 1])
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig(save_dir / 'grafico_taxa_sucesso.png', dpi=150, bbox_inches='tight')
    plt.close()

    # 5. Colisões por episódio
    plt.figure(figsize=(12, 6))
    plt.plot(episodes, metrics['collisions'], 'red', alpha=0.5, linewidth=1)
    if len(metrics['collisions']) >= 100:
        moving_avg = np.convolve(metrics['collisions'], np.ones(100)/100, mode='valid')
        plt.plot(range(100, len(metrics['collisions']) + 1), moving_avg, 'r-', linewidth=2, label='Média Móvel (100)')
    plt.xlabel('Episódio')
    plt.ylabel('Colisões')
    plt.title('Colisões por Episódio')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig(save_dir / 'grafico_colisoes.png', dpi=150, bbox_inches='tight')
    plt.close()

    # 6. Falhas por episódio
    plt.figure(figsize=(12, 6))
    failures_total = np.array(metrics['failures_r1']) + np.array(metrics['failures_r2'])
    plt.plot(episodes, failures_total, 'brown', alpha=0.5, linewidth=1, label='Total')
    plt.plot(episodes, metrics['failures_r1'], 'red', alpha=0.5, linewidth=1, label='R1')
    plt.plot(episodes, metrics['failures_r2'], 'blue', alpha=0.5, linewidth=1, label='R2')
    if len(failures_total) >= 100:
        moving_avg = np.convolve(failures_total, np.ones(100)/100, mode='valid')
        plt.plot(range(100, len(failures_total) + 1), moving_avg, 'r-', linewidth=2, label='Média Móvel Total')
    plt.xlabel('Episódio')
    plt.ylabel('Falhas')
    plt.title('Falhas por Episódio')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig(save_dir / 'grafico_falhas.png', dpi=150, bbox_inches='tight')
    plt.close()


def save_metrics_csv(metrics, save_dir):
    """Salva as métricas em arquivo CSV"""
    df = pd.DataFrame({
        'episodio': range(1, len(metrics['episode_rewards']) + 1),
        'recompensa': metrics['episode_rewards'],
        'entregas': metrics['episode_deliveries'],
        'steps': metrics['episode_steps'],
        'taxa_sucesso': metrics['success_rates'],
        'colisoes': metrics['collisions'],
        'falha_r1': metrics['failures_r1'],
        'falha_r2': metrics['failures_r2']
    })
    df.to_csv(save_dir / 'metricas_treinamento.csv', index=False)
    print(f"📊 Métricas salvas em: {save_dir / 'metricas_treinamento.csv'}")


def save_movement_log(movement_log, save_dir):
    """Salva o log de movimentos em arquivo CSV"""
    if movement_log:
        df = pd.DataFrame(movement_log)
        df.to_csv(save_dir / 'log_movimentos.csv', index=False)
        print(f"📊 Log de movimentos salvo em: {save_dir / 'log_movimentos.csv'}")


# ==================== FUNÇÃO DE TREINAMENTO ====================
def train_single_session(session_id, num_episodes, config, agents, trainer, save_dir):
    env = WarehouseEnv(config=config)

    metrics = {
        'episode_rewards': [],
        'episode_deliveries': [],
        'episode_steps': [],
        'success_rates': [],
        'collisions': [],
        'failures_r1': [],
        'failures_r2': [],
        'qmix_losses': [],
    }

    all_movement_logs = []

    best_reward = -float('inf')
    session_start_time = time.time()

    print(f"\n{'='*60}")
    print(f"🎯 SESSÃO QMIX COM FALHAS {session_id} - {num_episodes} episódios")
    print(f"   ⚠️ Probabilidade de falha: {config.FAILURE_PROBABILITY*100:.0f}%")
    print(f"{'='*60}")

    for episode in range(num_episodes):
        obs, _ = env.reset()
        episode_reward = 0
        episode_collisions = 0

        for agent in agents:
            agent.reset_failure_stats()

        global_state = env.get_global_state()

        for step in range(config.MAX_STEPS):
            actions = [agent.select_action(obs) for agent in agents]
            next_obs, rewards, terminated, truncated, info = env.step(actions)
            next_global_state = env.get_global_state()

            trainer.remember(obs, actions, rewards, next_obs, terminated or truncated,
                           global_state, next_global_state)

            episode_reward += sum(rewards)
            episode_collisions = info['collisions']

            qmix_loss = trainer.optimize()
            if qmix_loss and qmix_loss > 0:
                metrics['qmix_losses'].append(qmix_loss)

            obs = next_obs
            global_state = next_global_state

            if terminated or truncated:
                break

        metrics['episode_rewards'].append(episode_reward)
        metrics['episode_deliveries'].append(info['total_deliveries'])
        metrics['episode_steps'].append(step + 1)
        metrics['success_rates'].append(info['success_rate'])
        metrics['collisions'].append(episode_collisions)
        metrics['failures_r1'].append(info['failures_r1'])
        metrics['failures_r2'].append(info['failures_r2'])

        # Coletar logs de movimento
        all_movement_logs.extend(env.get_movement_log())

        if (episode + 1) % 100 == 0:
            recent_rewards = metrics['episode_rewards'][-100:]
            recent_deliveries = metrics['episode_deliveries'][-100:]
            epsilon = agents[0].get_epsilon()
            elapsed = time.time() - session_start_time

            print(f"  Sessão {session_id} | Ep {episode+1:4d}/{num_episodes} | "
                  f"Reward: {np.mean(recent_rewards):7.2f} | "
                  f"Entregas: {np.mean(recent_deliveries):.2f} | "
                  f"ε: {epsilon:.3f} | "
                  f"Tempo: {elapsed/60:.1f}min")

        if (episode + 1) % config.CLEAN_MEMORY_EVERY == 0:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    env.close()

    # Salvar logs de movimento da sessão
    save_movement_log(all_movement_logs, save_dir)

    session_stats = {
        'session_id': session_id,
        'total_episodes': num_episodes,
        'best_reward': max(metrics['episode_rewards']),
        'avg_reward_last_100': np.mean(metrics['episode_rewards'][-100:]),
        'avg_deliveries_last_100': np.mean(metrics['episode_deliveries'][-100:]),
        'final_success_rate': metrics['success_rates'][-1],
        'total_failures': sum(metrics['failures_r1'] + metrics['failures_r2']),
        'total_time_minutes': (time.time() - session_start_time) / 60
    }

    return metrics, session_stats


# ==================== FUNÇÃO PRINCIPAL ====================
def run_multi_session_training(num_sessions=2, episodes_per_session=1500):
    config = Config()
    config.EPISODES_PER_SESSION = episodes_per_session

    print("=" * 80)
    print("🏭 TREINAMENTO QMIX COM FALHAS - WAREHOUSE SEM OBSTÁCULOS Y")
    print("=" * 80)
    print(f"\n📋 CONFIGURAÇÃO:")
    print(f"   • Algoritmo: QMIX (Q-Mixing) com Falhas")
    print(f"   • Dispositivo: {'CUDA' if torch.cuda.is_available() else 'CPU'}")
    print(f"   • Sessões: {num_sessions}")
    print(f"   • Episódios por sessão: {episodes_per_session}")
    print(f"   • Total de episódios: {num_sessions * episodes_per_session}")
    print(f"   • ⚠️ PROBABILIDADE DE FALHA: {config.FAILURE_PROBABILITY*100:.0f}%")
    print("=" * 80)

    temp_env = WarehouseEnv(config=config)
    sample_obs, _ = temp_env.reset()
    state_dim = len(sample_obs)
    action_dim = temp_env.action_space[0].n
    global_state_dim = len(temp_env.get_global_state())
    temp_env.close()

    print(f"\n📊 Dimensões:")
    print(f"   • Estado individual: {state_dim}")
    print(f"   • Estado global: {global_state_dim}")
    print(f"   • Ação: {action_dim}")
    print(f"   • Robôs: 2")
    print(f"   • Caixas: 8")

    agents = [QMIXAgent(i, state_dim, action_dim, config, 2, global_state_dim) for i in range(2)]

    mixer = QMixer(n_agents=2, state_dim=global_state_dim, hidden_dim=config.MIXER_HIDDEN_DIM).to(agents[0].device)

    trainer = QMIXTrainer(agents, mixer, config, state_dim, global_state_dim)

    all_metrics = []
    all_session_stats = []
    consolidated_metrics = {
        'episode_rewards': [],
        'episode_deliveries': [],
        'episode_steps': [],
        'success_rates': [],
        'collisions': [],
        'failures_r1': [],
        'failures_r2': []
    }

    # Criar diretório de resultados
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    results_dir = Path(f"qmix_failure_results_{timestamp}")
    results_dir.mkdir(exist_ok=True)

    total_start_time = time.time()

    for session_id in range(1, num_sessions + 1):
        session_metrics, session_stats = train_single_session(
            session_id, episodes_per_session, config, agents, trainer, results_dir
        )

        all_metrics.append(session_metrics)
        all_session_stats.append(session_stats)

        for key in consolidated_metrics:
            consolidated_metrics[key].extend(session_metrics[key])

        print(f"\n📊 RESUMO SESSÃO {session_id}:")
        print(f"   • Melhor reward: {session_stats['best_reward']:.2f}")
        print(f"   • Média entregas (últimos 100): {session_stats['avg_deliveries_last_100']:.2f}")
        print(f"   • Taxa sucesso final: {session_stats['final_success_rate']:.1%}")
        print(f"   • Total falhas: {session_stats['total_failures']}")
        print(f"   • Tempo: {session_stats['total_time_minutes']:.1f} min")

    total_time = (time.time() - total_start_time) / 60

    print(f"\n💾 SALVANDO RESULTADOS...")

    # Salvar métricas consolidadas em CSV
    save_metrics_csv(consolidated_metrics, results_dir)

    # Plotar gráficos individuais
    plot_individual_graphs(consolidated_metrics, results_dir)

    # Salvar estatísticas das sessões
    df_stats = pd.DataFrame(all_session_stats)
    df_stats.to_csv(results_dir / "session_stats.csv", index=False)

    # Salvar modelos finais
    models_dir = results_dir / "models"
    models_dir.mkdir(exist_ok=True)
    for i, agent in enumerate(agents):
        torch.save(agent.policy_net.state_dict(), models_dir / f"qmix_agent_{i}_final.pth")
    torch.save(mixer.state_dict(), models_dir / "qmix_mixer_final.pth")

    # Salvar relatório final
    report = f"""
    ========================================
    RELATÓRIO FINAL DO TREINAMENTO QMIX
    ========================================

    DATA: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}

    CONFIGURAÇÃO:
    - Total de sessões: {len(all_session_stats)}
    - Total de episódios: {len(consolidated_metrics['episode_rewards'])}
    - Tempo total: {total_time:.1f} minutos
    - Probabilidade de falha: {config.FAILURE_PROBABILITY*100:.0f}%

    MÉTRICAS GERAIS:
    - Melhor recompensa: {max(consolidated_metrics['episode_rewards']):.2f}
    - Média recompensa (últimos 100): {np.mean(consolidated_metrics['episode_rewards'][-100:]):.2f}

    ENTREGAS:
    - Média entregas (últimos 100): {np.mean(consolidated_metrics['episode_deliveries'][-100:]):.2f}/8
    - Taxa de sucesso final: {consolidated_metrics['success_rates'][-1]:.1%}

    FALHAS:
    - Total de falhas: {sum(consolidated_metrics['failures_r1']) + sum(consolidated_metrics['failures_r2'])}
    - Média falhas por episódio: {np.mean(np.array(consolidated_metrics['failures_r1']) + np.array(consolidated_metrics['failures_r2'])):.2f}

    ========================================
    """

    with open(results_dir / "relatorio_final.txt", 'w', encoding='utf-8') as f:
        f.write(report)

    print(f"\n✅ TREINAMENTO CONCLUÍDO!")
    print(f"📁 Resultados salvos em: {results_dir}")
    print(f"⏱️ Tempo total: {total_time:.1f} minutos")
    print(f"\n📦 Arquivos gerados:")
    print(f"   • metricas_treinamento.csv - Métricas por episódio")
    print(f"   • log_movimentos.csv - Log de movimentos dos robôs")
    print(f"   • session_stats.csv - Estatísticas por sessão")
    print(f"   • grafico_recompensa.png - Gráfico de recompensa")
    print(f"   • grafico_entregas.png - Gráfico de entregas")
    print(f"   • grafico_steps.png - Gráfico de steps")
    print(f"   • grafico_taxa_sucesso.png - Gráfico de taxa de sucesso")
    print(f"   • grafico_colisoes.png - Gráfico de colisões")
    print(f"   • grafico_falhas.png - Gráfico de falhas")
    print(f"   • relatorio_final.txt - Relatório completo")
    print(f"   • models/ - Modelos treinados")

    return agents, mixer, consolidated_metrics


# ==================== EXECUÇÃO ====================
if __name__ == "__main__":
    NUM_SESSIONS = 1  # Reduzido para teste

    print("\n" + "=" * 80)
    print("🎮 INICIANDO TREINAMENTO QMIX COM FALHAS")
    print("=" * 80)

    try:
        agents, mixer, metrics = run_multi_session_training(
            num_sessions=NUM_SESSIONS,
            episodes_per_session=100  # Reduzido para teste rápido
        )

        print("\n✨ TREINAMENTO CONCLUÍDO COM SUCESSO! ✨")

    except Exception as e:
        print(f"\n❌ ERRO DURANTE O TREINAMENTO: {e}")
        import traceback
        traceback.print_exc()


🎮 INICIANDO TREINAMENTO QMIX COM FALHAS
🏭 TREINAMENTO QMIX COM FALHAS - WAREHOUSE SEM OBSTÁCULOS Y

📋 CONFIGURAÇÃO:
   • Algoritmo: QMIX (Q-Mixing) com Falhas
   • Dispositivo: CUDA
   • Sessões: 1
   • Episódios por sessão: 100
   • Total de episódios: 100
   • ⚠️ PROBABILIDADE DE FALHA: 20%

📊 Dimensões:
   • Estado individual: 40
   • Estado global: 28
   • Ação: 6
   • Robôs: 2
   • Caixas: 8

🎯 SESSÃO QMIX COM FALHAS 1 - 100 episódios
   ⚠️ Probabilidade de falha: 20%
  Sessão 1 | Ep  100/100 | Reward:   20.82 | Entregas: 1.42 | ε: 0.050 | Tempo: 4.5min
📊 Log de movimentos salvo em: qmix_failure_results_20260608_200637/log_movimentos.csv

📊 RESUMO SESSÃO 1:
   • Melhor reward: 175.17
   • Média entregas (últimos 100): 1.42
   • Taxa sucesso final: 0.0%
   • Total falhas: 14074
   • Tempo: 4.5 min

💾 SALVANDO RESULTADOS...
📊 Métricas salvas em: qmix_failure_results_20260608_200637/metricas_treinamento.csv

✅ TREINAMENTO CONCLUÍDO!
📁 Resultados salvos em: qmix_failure_results_20260

In [ ]:
%ls

'Ambiente e Execução IDQN - Versão 1.1.0.py'
'Ambiente e Execução IDQN - Versão 1.2.0.py'
'Ambiente e Execução IDQN - Versão 1.3.0.py'
'Armazem - Experimento.ipynb'
 Certo-warehouse_results_20260605_165855/
'Executa - Experimento.ipynb'
'Executa IDQN - Versão 7.0.0.py'
 OKwarehouse_results_20260604_000105/
 resultados_treinamento_20260604_051803.zip


In [ ]:
# ===========================

In [ ]:
"""
VERSÃO QUE FUNCIONOU SEM VIDEO
"""

import numpy as np
import gymnasium as gym
from gymnasium import spaces
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from collections import deque
import random
from typing import List, Tuple, Dict, Optional
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from datetime import datetime
import json
import os
from pathlib import Path
import imageio
from tqdm import tqdm
import pickle
import warnings
import time
import gc

warnings.filterwarnings('ignore')

# ==================== CONFIGURAÇÃO ====================
MAP_CONFIG = {
    'height': 12,
    'width': 8,
    'grid': [
        ['R1', '0', '0', '0', '0', '0', '0', 'R2'],
        ['0', 'Y', 'A', 'A', 'A', 'A', '0', '0'],
        ['0', '0', 'A', 'A', 'A', 'A', '0', '0'],
        ['X', '0', '0', '0', 'X', '0', 'Y', '0'],
        ['0', 'Y', 'X', '0', '0', '0', '0', '0'],
        ['0', '0', '0', 'X', '0', 'Y', 'X', 'X'],
        ['0', 'X', '0', 'Y', '0', 'X', '0', '0'],
        ['0', '0', '0', 'X', '0', '0', '0', '0'],
        ['X', '0', 'Y', '0', '0', '0', 'X', '0'],
        ['X', '0', 'B', 'B', 'B', 'B', 'X', '0'],
        ['X', '0', 'B', 'B', 'B', 'B', 'Y', '0'],
        ['0', '0', '0', 'Y', '0', '0', '0', '0']
    ]
}

class Config:
    # Exploração
    EPSILON_START = 1.0
    EPSILON_END = 0.05
    EPSILON_DECAY_STEPS = 50000

    # Aprendizado
    LEARNING_RATE = 0.0001
    BATCH_SIZE = 256
    GAMMA = 0.95
    TAU = 0.001
    WEIGHT_DECAY = 1e-5

    # Rede Neural
    HIDDEN_DIM = 256  # Reduzido para memória
    DROPOUT_RATE = 0.2

    # Memória
    BUFFER_SIZE = 100000  # Reduzido para memória
    PRIORITIZED_REPLAY = True
    ALPHA = 0.6
    BETA = 0.4

    # Treinamento
    MAX_GRAD_NORM = 1.0
    LEARNING_STARTS = 1000
    TRAIN_FREQ = 4
    USE_SOFT_UPDATE = True
    MAX_STEPS = 500
    EPISODES_PER_SESSION = 1500

    # Sistema
    SAVE_INTERVAL = 100  # Salvar métricas a cada 100 episódios
    CLEAN_MEMORY_EVERY = 500  # Limpar memória a cada 500 episódios

# ==================== PRIORITIZED REPLAY BUFFER ====================
class PrioritizedReplayBuffer:
    def __init__(self, capacity, alpha=0.6, beta=0.4):
        self.capacity = capacity
        self.alpha = alpha
        self.beta = beta
        self.buffer = []
        self.priorities = []
        self.position = 0

    def push(self, state, action, reward, next_state, done):
        max_priority = max(self.priorities) if self.priorities else 1.0

        if len(self.buffer) < self.capacity:
            self.buffer.append((state, action, reward, next_state, done))
            self.priorities.append(max_priority)
        else:
            self.buffer[self.position] = (state, action, reward, next_state, done)
            self.priorities[self.position] = max_priority

        self.position = (self.position + 1) % self.capacity

    def sample(self, batch_size):
        if len(self.buffer) == self.capacity:
            priorities = np.array(self.priorities)
        else:
            priorities = np.array(self.priorities[:len(self.buffer)])

        probs = priorities ** self.alpha
        probs /= probs.sum()

        indices = np.random.choice(len(self.buffer), batch_size, p=probs)

        total = len(self.buffer)
        weights = (total * probs[indices]) ** (-self.beta)
        weights /= weights.max()

        batch = [self.buffer[idx] for idx in indices]
        states, actions, rewards, next_states, dones = zip(*batch)

        return (np.array(states), np.array(actions), np.array(rewards),
                np.array(next_states), np.array(dones), indices, weights)

    def update_priorities(self, indices, td_errors):
        for idx, td_error in zip(indices, td_errors):
            self.priorities[idx] = abs(td_error) + 1e-6

    def __len__(self):
        return len(self.buffer)

# ==================== REDE NEURAL ====================
class DQN(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_dim=256, dropout=0.2):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight)
            nn.init.constant_(module.bias, 0.0)

    def forward(self, x):
        return self.network(x)

# ==================== AGENTE IDQN ====================
class IDQNAgent:
    def __init__(self, state_dim, action_dim, agent_id, config):
        self.agent_id = agent_id
        self.action_dim = action_dim
        self.config = config
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        print(f"  Agente {agent_id} usando dispositivo: {self.device}")

        self.policy_net = DQN(state_dim, action_dim, config.HIDDEN_DIM, config.DROPOUT_RATE).to(self.device)
        self.target_net = DQN(state_dim, action_dim, config.HIDDEN_DIM, config.DROPOUT_RATE).to(self.device)
        self.target_net.load_state_dict(self.policy_net.state_dict())

        self.optimizer = optim.Adam(self.policy_net.parameters(),
                                   lr=config.LEARNING_RATE,
                                   weight_decay=config.WEIGHT_DECAY)

        self.memory = PrioritizedReplayBuffer(config.BUFFER_SIZE, alpha=config.ALPHA)

        self.steps_done = 0
        self.learning_steps = 0
        self.total_episodes = 0
        self.losses = []

    def get_epsilon(self):
        if self.steps_done >= self.config.EPSILON_DECAY_STEPS:
            return self.config.EPSILON_END

        epsilon = self.config.EPSILON_START - (self.config.EPSILON_START - self.config.EPSILON_END) * \
                  self.steps_done / self.config.EPSILON_DECAY_STEPS
        return max(self.config.EPSILON_END, epsilon)

    def select_action(self, state, training=True):
        self.steps_done += 1

        if training and random.random() < self.get_epsilon():
            return random.randrange(self.action_dim)

        with torch.no_grad():
            state_tensor = torch.FloatTensor(state).unsqueeze(0).to(self.device)
            q_values = self.policy_net(state_tensor)
            return q_values.argmax().item()

    def remember(self, state, action, reward, next_state, done):
        self.memory.push(state, action, reward, next_state, done)

    def optimize(self):
        if len(self.memory) < self.config.BATCH_SIZE or \
           self.steps_done < self.config.LEARNING_STARTS:
            return 0

        self.learning_steps += 1

        if self.learning_steps % self.config.TRAIN_FREQ != 0:
            return 0

        states, actions, rewards, next_states, dones, indices, weights = self.memory.sample(self.config.BATCH_SIZE)
        weights = torch.FloatTensor(weights).to(self.device)

        states = torch.FloatTensor(np.array(states)).to(self.device)
        actions = torch.LongTensor(np.array(actions)).to(self.device)
        rewards = torch.FloatTensor(np.array(rewards)).to(self.device)
        next_states = torch.FloatTensor(np.array(next_states)).to(self.device)
        dones = torch.FloatTensor(np.array(dones)).to(self.device)

        with torch.no_grad():
            next_actions = self.policy_net(next_states).argmax(1, keepdim=True)
            next_q_values = self.target_net(next_states).gather(1, next_actions).squeeze()
            target_q = rewards + self.config.GAMMA * next_q_values * (1 - dones)

        current_q = self.policy_net(states).gather(1, actions.unsqueeze(1)).squeeze()

        td_errors = target_q - current_q
        loss = (weights * td_errors.pow(2)).mean()

        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.policy_net.parameters(), self.config.MAX_GRAD_NORM)
        self.optimizer.step()

        priorities = td_errors.abs().detach().cpu().numpy() + 1e-6
        self.memory.update_priorities(indices, priorities)

        if self.config.USE_SOFT_UPDATE:
            self.soft_update_target()

        self.losses.append(loss.item())
        return loss.item()

    def soft_update_target(self):
        for target_param, policy_param in zip(self.target_net.parameters(),
                                               self.policy_net.parameters()):
            target_param.data.copy_(self.config.TAU * policy_param.data +
                                   (1 - self.config.TAU) * target_param.data)

# ==================== AMBIENTE WAREHOUSE ====================
class WarehouseEnv(gym.Env):
    metadata = {'render.modes': ['rgb_array']}

    def __init__(self, config=None):
        super().__init__()

        self.config = config or Config()
        self.height = MAP_CONFIG['height']
        self.width = MAP_CONFIG['width']
        self.grid = [row[:] for row in MAP_CONFIG['grid']]

        self.robot_positions = None
        self.box_positions = None
        self.targets = self._find_positions('B')

        self.num_robots = 2
        self.num_boxes = len(self._find_positions('A'))
        self.num_targets = len(self.targets)

        self.delivered_boxes = None
        self.steps = 0
        self.max_steps = self.config.MAX_STEPS
        self.total_deliveries = 0
        self.collisions = 0
        self.distance_traveled = [0, 0]
        self.previous_distances = None

        self.action_space = spaces.Tuple([spaces.Discrete(6) for _ in range(self.num_robots)])

        obs_dim = (self.num_robots * 2) + (self.num_boxes * 2) + (self.num_targets * 2) + 8
        self.observation_space = spaces.Box(
            low=-1, high=self.height + self.width,
            shape=(obs_dim,),
            dtype=np.float32
        )

        self.frame_buffer = []

    def _find_positions(self, symbols):
        if isinstance(symbols, str):
            symbols = [symbols]
        positions = []
        for i in range(self.height):
            for j in range(self.width):
                cell = self.grid[i][j]
                if any(cell.startswith(sym) for sym in symbols):
                    positions.append((i, j))
        return positions

    def reset(self):
        self.grid = [row[:] for row in MAP_CONFIG['grid']]
        self.robot_positions = self._find_positions('R')
        self.box_positions = self._find_positions('A')
        self.delivered_boxes = [False] * self.num_boxes

        self.steps = 0
        self.total_deliveries = 0
        self.collisions = 0
        self.distance_traveled = [0, 0]

        self.previous_distances = [self._min_distance_to_boxes(r) for r in range(self.num_robots)]

        return self._get_observation(), self._get_info()

    def _min_distance_to_boxes(self, robot_id):
        robot_pos = self.robot_positions[robot_id]
        remaining_boxes = [box_pos for box_pos in self.box_positions
                          if box_pos is not None and
                          not self.delivered_boxes[self.box_positions.index(box_pos)]]

        if not remaining_boxes:
            return 0
        return min([abs(robot_pos[0] - box_pos[0]) + abs(robot_pos[1] - box_pos[1])
                   for box_pos in remaining_boxes])

    def _is_valid_position(self, pos, robot_id=None):
        i, j = pos
        if i < 0 or i >= self.height or j < 0 or j >= self.width:
            return False

        cell = self.grid[i][j]
        if cell in ['X', 'Y']:
            return False

        if robot_id is not None:
            for rid, rpos in enumerate(self.robot_positions):
                if rid != robot_id and rpos == (i, j):
                    return False
        return True

    def _move_robot(self, robot_id, action):
        i, j = self.robot_positions[robot_id]
        new_i, new_j = i, j

        if action == 0: new_i -= 1
        elif action == 1: new_i += 1
        elif action == 2: new_j -= 1
        elif action == 3: new_j += 1

        if self._is_valid_position((new_i, new_j), robot_id):
            self.distance_traveled[robot_id] += 1
            old_pos = self.robot_positions[robot_id]
            self.grid[old_pos[0]][old_pos[1]] = '0'
            self.grid[new_i][new_j] = f'R{robot_id + 1}'
            self.robot_positions[robot_id] = (new_i, new_j)
            return -0.005
        else:
            self.collisions += 1
            return -0.02

    def _pickup_box(self, robot_id):
        robot_pos = self.robot_positions[robot_id]
        for box_id, box_pos in enumerate(self.box_positions):
            if not self.delivered_boxes[box_id] and box_pos == robot_pos:
                self.box_positions[box_id] = None
                self.grid[robot_pos[0]][robot_pos[1]] = f'R{robot_id + 1}'
                return 2.0
        return -0.02

    def _drop_box(self, robot_id):
        robot_pos = self.robot_positions[robot_id]

        box_with_robot = None
        for box_id, box_pos in enumerate(self.box_positions):
            if box_pos is None and not self.delivered_boxes[box_id]:
                box_with_robot = box_id
                break

        if box_with_robot is None:
            return -0.02

        for target_pos in self.targets:
            if robot_pos == target_pos:
                self.delivered_boxes[box_with_robot] = True
                self.total_deliveries += 1
                self.grid[robot_pos[0]][robot_pos[1]] = f'R{robot_id + 1}'
                return 25.0

        return -0.05

    def _calculate_shaped_reward(self, robot_id, base_reward):
        reward = base_reward

        current_distance = self._min_distance_to_boxes(robot_id)
        previous_distance = self.previous_distances[robot_id]

        if current_distance < previous_distance:
            reward += 0.1 * (previous_distance - current_distance)
        elif current_distance > previous_distance:
            reward -= 0.02 * (current_distance - previous_distance)

        self.previous_distances[robot_id] = current_distance

        if all(self.delivered_boxes):
            reward += 50.0

        return reward

    def _get_observation(self):
        obs = []

        for robot_pos in self.robot_positions:
            obs.append(robot_pos[0] / self.height)
            obs.append(robot_pos[1] / self.width)

        for box_id, box_pos in enumerate(self.box_positions):
            if box_pos is None or self.delivered_boxes[box_id]:
                obs.append(-1)
                obs.append(-1)
            else:
                obs.append(box_pos[0] / self.height)
                obs.append(box_pos[1] / self.width)

        for target_pos in self.targets:
            obs.append(target_pos[0] / self.height)
            obs.append(target_pos[1] / self.width)

        for robot_pos in self.robot_positions:
            min_box_dist = min([abs(robot_pos[0] - box_pos[0]) + abs(robot_pos[1] - box_pos[1])
                               for box_pos in self.box_positions
                               if box_pos is not None and
                               not self.delivered_boxes[self.box_positions.index(box_pos)]],
                              default=100)
            obs.append(min_box_dist / (self.height + self.width))

            min_target_dist = min([abs(robot_pos[0] - target_pos[0]) + abs(robot_pos[1] - target_pos[1])
                                  for target_pos in self.targets],
                                 default=100)
            obs.append(min_target_dist / (self.height + self.width))

        return np.array(obs, dtype=np.float32)

    def step(self, actions):
        self.steps += 1

        if len(actions) != self.num_robots:
            actions = [actions] * self.num_robots

        total_reward = 0
        rewards = [0, 0]

        # Movimentos
        movement_rewards = []
        for robot_id, action in enumerate(actions):
            if action < 4:
                reward = self._move_robot(robot_id, action)
                movement_rewards.append(reward)
            else:
                movement_rewards.append(0)

        # Interações
        interaction_rewards = []
        for robot_id, action in enumerate(actions):
            if action == 4:
                reward = self._pickup_box(robot_id)
                interaction_rewards.append(reward)
            elif action == 5:
                reward = self._drop_box(robot_id)
                interaction_rewards.append(reward)
            else:
                interaction_rewards.append(0)

        for robot_id in range(self.num_robots):
            base_reward = movement_rewards[robot_id] + interaction_rewards[robot_id]
            shaped_reward = self._calculate_shaped_reward(robot_id, base_reward)
            rewards[robot_id] = shaped_reward
            total_reward += shaped_reward

        terminated = all(self.delivered_boxes)
        truncated = self.steps >= self.max_steps

        observation = self._get_observation()
        info = self._get_info()

        return observation, rewards, terminated, truncated, info

    def _get_info(self):
        return {
            'steps': self.steps,
            'total_deliveries': self.total_deliveries,
            'collisions': self.collisions,
            'distance_traveled': self.distance_traveled.copy(),
            'remaining_boxes': sum(1 for d in self.delivered_boxes if not d),
            'success_rate': self.total_deliveries / self.num_boxes if self.steps > 0 else 0,
        }

    def render(self):
        fig, ax = plt.subplots(figsize=(10, 8))
        self._draw_grid(ax)
        fig.canvas.draw()
        img = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
        img = img.reshape(fig.canvas.get_width_height()[::-1] + (3,))
        plt.close(fig)
        return img

    def _draw_grid(self, ax):
        colors = {
            '0': 'white', 'X': 'black', 'Y': 'gray',
            'R1': 'red', 'R2': 'blue', 'A': 'orange', 'B': 'green'
        }

        for i in range(self.height):
            for j in range(self.width):
                cell = self.grid[i][j]
                color = colors.get(cell, 'white')
                rect = Rectangle((j, self.height - 1 - i), 1, 1,
                               facecolor=color, edgecolor='black', linewidth=0.5)
                ax.add_patch(rect)

                if cell in ['R1', 'R2', 'A', 'B']:
                    ax.text(j + 0.5, self.height - 0.5 - i, cell,
                           ha='center', va='center', fontweight='bold')

        ax.set_xlim(0, self.width)
        ax.set_ylim(0, self.height)
        ax.set_xticks(range(self.width))
        ax.set_yticks(range(self.height))
        ax.grid(True, alpha=0.3)
        ax.set_title(f'Warehouse | Steps: {self.steps} | Entregas: {self.total_deliveries}/4')

    def close(self):
        pass

# ==================== FUNÇÃO DE TREINAMENTO ====================
def train_single_session(session_id, num_episodes, config, agents):
    """Treina uma única sessão"""

    env = WarehouseEnv(config=config)

    metrics = {
        'episode_rewards': [],
        'episode_deliveries': [],
        'episode_steps': [],
        'success_rates': [],
        'collisions': [],
    }

    best_reward = -float('inf')
    session_start_time = time.time()

    print(f"\n{'='*60}")
    print(f"🎯 SESSÃO {session_id} - {num_episodes} episódios")
    print(f"{'='*60}")

    for episode in range(num_episodes):
        obs, _ = env.reset()
        episode_reward = 0
        episode_collisions = 0

        for step in range(config.MAX_STEPS):
            actions = [agent.select_action(obs) for agent in agents]
            next_obs, rewards, terminated, truncated, info = env.step(actions)

            for i, agent in enumerate(agents):
                agent.remember(obs, actions[i], rewards[i], next_obs, terminated or truncated)
                episode_reward += rewards[i]

            episode_collisions = info['collisions']

            for agent in agents:
                agent.optimize()

            obs = next_obs

            if terminated or truncated:
                break

        # Registrar métricas
        metrics['episode_rewards'].append(episode_reward)
        metrics['episode_deliveries'].append(info['total_deliveries'])
        metrics['episode_steps'].append(step + 1)
        metrics['success_rates'].append(info['success_rate'])
        metrics['collisions'].append(episode_collisions)

        # Logging
        if (episode + 1) % 100 == 0:
            recent_rewards = metrics['episode_rewards'][-100:]
            recent_deliveries = metrics['episode_deliveries'][-100:]
            epsilon = agents[0].get_epsilon()
            elapsed = time.time() - session_start_time

            print(f"  Sessão {session_id} | Ep {episode+1:4d}/{num_episodes} | "
                  f"Reward: {np.mean(recent_rewards):7.2f} | "
                  f"Entregas: {np.mean(recent_deliveries):.2f} | "
                  f"ε: {epsilon:.3f} | "
                  f"Tempo: {elapsed/60:.1f}min")

        # Limpar memória periodicamente
        if (episode + 1) % config.CLEAN_MEMORY_EVERY == 0:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    env.close()

    # Calcular estatísticas da sessão
    session_stats = {
        'session_id': session_id,
        'total_episodes': num_episodes,
        'best_reward': max(metrics['episode_rewards']),
        'avg_reward_last_100': np.mean(metrics['episode_rewards'][-100:]),
        'avg_deliveries_last_100': np.mean(metrics['episode_deliveries'][-100:]),
        'final_success_rate': metrics['success_rates'][-1],
        'total_time_minutes': (time.time() - session_start_time) / 60
    }

    return metrics, session_stats

# ==================== FUNÇÃO PRINCIPAL ====================
def run_multi_session_training(num_sessions=4, episodes_per_session=1500):
    """Executa treinamento em múltiplas sessões"""

    config = Config()
    config.EPISODES_PER_SESSION = episodes_per_session

    print("=" * 80)
    print("🏭 TREINAMENTO MULTI-SESSÃO - WAREHOUSE COM 2 ROBÔS")
    print("=" * 80)
    print(f"\n📋 CONFIGURAÇÃO:")
    print(f"   • Dispositivo: {'CUDA' if torch.cuda.is_available() else 'CPU'}")
    print(f"   • Sessões: {num_sessions}")
    print(f"   • Episódios por sessão: {episodes_per_session}")
    print(f"   • Total de episódios: {num_sessions * episodes_per_session}")
    print(f"   • Batch Size: {config.BATCH_SIZE}")
    print(f"   • Learning Rate: {config.LEARNING_RATE}")
    print(f"   • Hidden Dim: {config.HIDDEN_DIM}")
    print("=" * 80)

    # Inicializar ambiente para obter dimensões
    temp_env = WarehouseEnv(config=config)
    sample_obs, _ = temp_env.reset()
    state_dim = len(sample_obs)
    action_dim = temp_env.action_space[0].n
    temp_env.close()

    print(f"\n📊 Dimensões:")
    print(f"   • Estado: {state_dim}")
    print(f"   • Ação: {action_dim}")
    print(f"   • Robôs: 2")
    print(f"   • Caixas: 4")

    # Criar agentes
    agents = [IDQNAgent(state_dim, action_dim, i, config) for i in range(2)]

    all_metrics = []
    all_session_stats = []
    consolidated_metrics = {
        'episode_rewards': [],
        'episode_deliveries': [],
        'episode_steps': [],
        'success_rates': [],
        'collisions': []
    }

    total_start_time = time.time()

    for session_id in range(1, num_sessions + 1):
        # Treinar sessão
        session_metrics, session_stats = train_single_session(
            session_id, episodes_per_session, config, agents
        )

        all_metrics.append(session_metrics)
        all_session_stats.append(session_stats)

        # Consolidar métricas
        for key in consolidated_metrics:
            consolidated_metrics[key].extend(session_metrics[key])

        # Mostrar resumo da sessão
        print(f"\n📊 RESUMO SESSÃO {session_id}:")
        print(f"   • Melhor reward: {session_stats['best_reward']:.2f}")
        print(f"   • Média entregas (últimos 100): {session_stats['avg_deliveries_last_100']:.2f}")
        print(f"   • Taxa sucesso final: {session_stats['final_success_rate']:.1%}")
        print(f"   • Tempo: {session_stats['total_time_minutes']:.1f} min")

    total_time = (time.time() - total_start_time) / 60

    # Salvar resultados
    print(f"\n💾 SALVANDO RESULTADOS...")

    # Criar diretório de resultados
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    results_dir = Path(f"warehouse_results_{timestamp}")
    results_dir.mkdir(exist_ok=True)

    # Salvar métricas consolidadas
    df_consolidated = pd.DataFrame({
        'episode': range(1, len(consolidated_metrics['episode_rewards']) + 1),
        'reward': consolidated_metrics['episode_rewards'],
        'deliveries': consolidated_metrics['episode_deliveries'],
        'steps': consolidated_metrics['episode_steps'],
        'success_rate': consolidated_metrics['success_rates'],
        'collisions': consolidated_metrics['collisions']
    })
    df_consolidated.to_csv(results_dir / "consolidated_metrics.csv", index=False)

    # Salvar estatísticas das sessões
    df_stats = pd.DataFrame(all_session_stats)
    df_stats.to_csv(results_dir / "session_stats.csv", index=False)

    # Salvar modelos finais
    models_dir = results_dir / "models"
    models_dir.mkdir(exist_ok=True)
    for i, agent in enumerate(agents):
        torch.save(agent.policy_net.state_dict(), models_dir / f"agent_{i}_final.pth")

    # Plotar gráficos
    plot_training_results(consolidated_metrics, all_session_stats, results_dir)

    # Criar relatório final
    create_final_report(consolidated_metrics, all_session_stats, total_time, results_dir)

    print(f"\n✅ TREINAMENTO CONCLUÍDO!")
    print(f"📁 Resultados salvos em: {results_dir}")
    print(f"⏱️ Tempo total: {total_time:.1f} minutos")

    # Tentar fazer download dos arquivos
    try:
        from google.colab import files
        print("\n📥 Fazendo download dos resultados...")
        for file_path in results_dir.glob("*"):
            files.download(str(file_path))
        for file_path in models_dir.glob("*"):
            files.download(str(file_path))
    except:
        print("\n⚠️ Não foi possível fazer download automático. Os arquivos estão em:", results_dir)

    return agents, consolidated_metrics

def plot_training_results(metrics, session_stats, save_dir):
    """Plota gráficos de treinamento"""

    fig, axes = plt.subplots(2, 3, figsize=(15, 10))

    # 1. Recompensa
    axes[0, 0].plot(metrics['episode_rewards'], alpha=0.3, linewidth=0.5)
    if len(metrics['episode_rewards']) >= 100:
        moving_avg = np.convolve(metrics['episode_rewards'], np.ones(100)/100, mode='valid')
        axes[0, 0].plot(range(99, len(metrics['episode_rewards'])), moving_avg, 'r-', linewidth=2)
    axes[0, 0].set_title('Recompensa por Episódio')
    axes[0, 0].set_xlabel('Episódio')
    axes[0, 0].set_ylabel('Recompensa')
    axes[0, 0].grid(True, alpha=0.3)

    # 2. Entregas
    axes[0, 1].plot(metrics['episode_deliveries'], alpha=0.3, linewidth=0.5, color='green')
    if len(metrics['episode_deliveries']) >= 100:
        moving_avg_del = np.convolve(metrics['episode_deliveries'], np.ones(100)/100, mode='valid')
        axes[0, 1].plot(range(99, len(metrics['episode_deliveries'])), moving_avg_del, 'r-', linewidth=2)
    axes[0, 1].axhline(y=4, color='g', linestyle='--', label='Meta')
    axes[0, 1].set_title('Entregas por Episódio')
    axes[0, 1].set_xlabel('Episódio')
    axes[0, 1].set_ylabel('Entregas')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

    # 3. Steps
    axes[0, 2].plot(metrics['episode_steps'], color='orange', alpha=0.5)
    axes[0, 2].set_title('Steps por Episódio')
    axes[0, 2].set_xlabel('Episódio')
    axes[0, 2].set_ylabel('Steps')
    axes[0, 2].grid(True, alpha=0.3)

    # 4. Taxa de Sucesso
    axes[1, 0].plot(metrics['success_rates'], color='purple', alpha=0.5)
    if len(metrics['success_rates']) >= 100:
        moving_avg_success = np.convolve(metrics['success_rates'], np.ones(100)/100, mode='valid')
        axes[1, 0].plot(range(99, len(metrics['success_rates'])), moving_avg_success, 'r-', linewidth=2)
    axes[1, 0].axhline(y=0.95, color='g', linestyle='--', label='Meta 95%')
    axes[1, 0].set_title('Taxa de Sucesso')
    axes[1, 0].set_xlabel('Episódio')
    axes[1, 0].set_ylabel('Taxa')
    axes[1, 0].set_ylim([0, 1])
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

    # 5. Colisões
    axes[1, 1].plot(metrics['collisions'], color='red', alpha=0.5)
    if len(metrics['collisions']) >= 100:
        moving_avg_coll = np.convolve(metrics['collisions'], np.ones(100)/100, mode='valid')
        axes[1, 1].plot(range(99, len(metrics['collisions'])), moving_avg_coll, 'r-', linewidth=2)
    axes[1, 1].set_title('Colisões por Episódio')
    axes[1, 1].set_xlabel('Episódio')
    axes[1, 1].set_ylabel('Colisões')
    axes[1, 1].grid(True, alpha=0.3)

    # 6. Progresso por Sessão
    axes[1, 2].bar(range(1, len(session_stats) + 1),
                   [s['avg_deliveries_last_100'] for s in session_stats],
                   color='blue', alpha=0.7)
    axes[1, 2].axhline(y=4, color='g', linestyle='--', label='Meta')
    axes[1, 2].set_title('Média de Entregas por Sessão')
    axes[1, 2].set_xlabel('Sessão')
    axes[1, 2].set_ylabel('Entregas Médias')
    axes[1, 2].legend()
    axes[1, 2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_dir / "training_results.png", dpi=150)
    plt.close()

def create_final_report(metrics, session_stats, total_time, save_dir):
    """Cria relatório final"""

    report = f"""
    ========================================
    RELATÓRIO FINAL DO TREINAMENTO
    ========================================

    DATA: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}

    CONFIGURAÇÃO:
    - Total de sessões: {len(session_stats)}
    - Total de episódios: {len(metrics['episode_rewards'])}
    - Tempo total: {total_time:.1f} minutos

    MÉTRICAS GERAIS:
    - Melhor recompensa: {max(metrics['episode_rewards']):.2f}
    - Média recompensa (últimos 100): {np.mean(metrics['episode_rewards'][-100:]):.2f}

    ENTREGAS:
    - Média entregas (últimos 100): {np.mean(metrics['episode_deliveries'][-100:]):.2f}
    - Máximo entregas: {max(metrics['episode_deliveries'])}
    - Taxa de sucesso final: {metrics['success_rates'][-1]:.1%}

    EFICIÊNCIA:
    - Média steps (últimos 100): {np.mean(metrics['episode_steps'][-100:]):.1f}
    - Média colisões (últimos 100): {np.mean(metrics['collisions'][-100:]):.1f}

    PROGRESSO POR SESSÃO:
    """

    for stat in session_stats:
        report += f"""
    Sessão {stat['session_id']}:
        - Melhor reward: {stat['best_reward']:.2f}
        - Média entregas: {stat['avg_deliveries_last_100']:.2f}
        - Taxa sucesso final: {stat['final_success_rate']:.1%}
        - Tempo: {stat['total_time_minutes']:.1f} min
    """

    report += """
    ========================================
    """

    with open(save_dir / "final_report.txt", 'w') as f:
        f.write(report)

    print(report)

# ==================== EXECUÇÃO ====================
if __name__ == "__main__":
    # Configurar número de sessões (cada sessão = 1500 episódios)
    NUM_SESSIONS = 2  # Começar com 2 sessões para teste

    print("\n" + "=" * 80)
    print("🎮 INICIANDO TREINAMENTO DO WAREHOUSE COM 2 ROBÔS")
    print("=" * 80)

    try:
        # Executar treinamento
        agents, metrics = run_multi_session_training(
            num_sessions=NUM_SESSIONS,
            episodes_per_session=1500
        )

        print("\n✨ TREINAMENTO CONCLUÍDO COM SUCESSO! ✨")

    except Exception as e:
        print(f"\n❌ ERRO DURANTE O TREINAMENTO: {e}")
        import traceback
        traceback.print_exc()


🎮 INICIANDO TREINAMENTO DO WAREHOUSE COM 2 ROBÔS
🏭 TREINAMENTO MULTI-SESSÃO - WAREHOUSE COM 2 ROBÔS

📋 CONFIGURAÇÃO:
   • Dispositivo: CUDA
   • Sessões: 2
   • Episódios por sessão: 1500
   • Total de episódios: 3000
   • Batch Size: 256
   • Learning Rate: 0.0001
   • Hidden Dim: 256

📊 Dimensões:
   • Estado: 40
   • Ação: 6
   • Robôs: 2
   • Caixas: 4
  Agente 0 usando dispositivo: cuda
  Agente 1 usando dispositivo: cuda

🎯 SESSÃO 1 - 1500 episódios
  Sessão 1 | Ep  100/1500 | Reward:  161.79 | Entregas: 5.52 | ε: 0.092 | Tempo: 4.1min
  Sessão 1 | Ep  200/1500 | Reward:   93.00 | Entregas: 3.67 | ε: 0.050 | Tempo: 14.3min
  Sessão 1 | Ep  300/1500 | Reward:   86.22 | Entregas: 3.48 | ε: 0.050 | Tempo: 27.8min
  Sessão 1 | Ep  400/1500 | Reward:  102.06 | Entregas: 3.96 | ε: 0.050 | Tempo: 41.3min
  Sessão 1 | Ep  500/1500 | Reward:  123.31 | Entregas: 4.59 | ε: 0.050 | Tempo: 54.5min
  Sessão 1 | Ep  600/1500 | Reward:  108.89 | Entregas: 4.12 | ε: 0.050 | Tempo: 68.3min
  Sess

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✨ TREINAMENTO CONCLUÍDO COM SUCESSO! ✨


In [ ]:
# faz video dos robos
"""
PROGRAMA PARA VISUALIZAR ROBÔS TREINADOS COM IDQN
Carrega os modelos agent_0_final.pth e agent_1_final.pth
Gera vídeo da movimentação dos robôs no warehouse
"""

import numpy as np
import gymnasium as gym
from gymnasium import spaces
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.animation import FuncAnimation, FFMpegWriter
import torch
import torch.nn as nn
from pathlib import Path
import imageio
from typing import List, Tuple, Dict
import warnings
import os

warnings.filterwarnings('ignore')

# ==================== CONFIGURAÇÃO DO AMBIENTE ====================
MAP_CONFIG = {
    'height': 12,
    'width': 8,
    'grid': [
        ['R1', '0', '0', '0', '0', '0', '0', 'R2'],
        ['0', 'Y', 'A', 'A', 'A', 'A', '0', '0'],
        ['0', '0', 'A', 'A', 'A', 'A', '0', '0'],
        ['X', '0', '0', '0', 'X', '0', 'Y', '0'],
        ['0', 'Y', 'X', '0', '0', '0', '0', '0'],
        ['0', '0', '0', 'X', '0', 'Y', 'X', 'X'],
        ['0', 'X', '0', 'Y', '0', 'X', '0', '0'],
        ['0', '0', '0', 'X', '0', '0', '0', '0'],
        ['X', '0', 'Y', '0', '0', '0', 'X', '0'],
        ['X', '0', 'B', 'B', 'B', 'B', 'X', '0'],
        ['X', '0', 'B', 'B', 'B', 'B', 'Y', '0'],
        ['0', '0', '0', 'Y', '0', '0', '0', '0']
    ]
}

class Config:
    HIDDEN_DIM = 256
    DROPOUT_RATE = 0.2
    MAX_STEPS = 500

# ==================== REDE NEURAL (MESMA ARQUITETURA DO TREINAMENTO) ====================
class DQN(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_dim=256, dropout=0.2):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        return self.network(x)

# ==================== AGENTE (MESMA ESTRUTURA DO TREINAMENTO) ====================
class IDQNAgent:
    def __init__(self, state_dim, action_dim, agent_id):
        self.agent_id = agent_id
        self.action_dim = action_dim
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        self.policy_net = DQN(state_dim, action_dim, Config.HIDDEN_DIM, Config.DROPOUT_RATE).to(self.device)

    def load_model(self, model_path):
        """Carrega o modelo treinado"""
        try:
            self.policy_net.load_state_dict(torch.load(model_path, map_location=self.device))
            print(f"✅ Modelo do Agente {self.agent_id} carregado de: {model_path}")
            return True
        except Exception as e:
            print(f"❌ Erro ao carregar modelo do Agente {self.agent_id}: {e}")
            return False

    def select_action(self, state):
        """Seleciona ação sem exploração (modo avaliação)"""
        with torch.no_grad():
            state_tensor = torch.FloatTensor(state).unsqueeze(0).to(self.device)
            q_values = self.policy_net(state_tensor)
            return q_values.argmax().item()

# ==================== AMBIENTE WAREHOUSE (VERSÃO SIMPLIFICADA PARA VISUALIZAÇÃO) ====================
class WarehouseEnv:
    def __init__(self):
        self.height = MAP_CONFIG['height']
        self.width = MAP_CONFIG['width']
        self.grid = [row[:] for row in MAP_CONFIG['grid']]
        self.max_steps = Config.MAX_STEPS

    def reset(self):
        """Reinicia o ambiente"""
        self.grid = [row[:] for row in MAP_CONFIG['grid']]
        self.robot_positions = self._find_positions(['R1', 'R2'])
        self.box_positions = self._find_positions(['A'])
        self.delivered_boxes = [False] * len(self.box_positions)
        self.targets = self._find_positions(['B'])

        self.steps = 0
        self.total_deliveries = 0
        self.collisions = 0

        return self._get_observation()

    def _find_positions(self, symbols):
        """Encontra posições de símbolos no grid"""
        positions = []
        for i in range(self.height):
            for j in range(self.width):
                cell = self.grid[i][j]
                if cell in symbols:
                    positions.append((i, j))
        return positions

    def _get_observation(self):
        """Gera observação para os agentes (mesma estrutura do treinamento)"""
        obs = []

        for robot_pos in self.robot_positions:
            obs.append(robot_pos[0] / self.height)
            obs.append(robot_pos[1] / self.width)

        for box_id, box_pos in enumerate(self.box_positions):
            if box_pos is None or self.delivered_boxes[box_id]:
                obs.append(-1)
                obs.append(-1)
            else:
                obs.append(box_pos[0] / self.height)
                obs.append(box_pos[1] / self.width)

        for target_pos in self.targets:
            obs.append(target_pos[0] / self.height)
            obs.append(target_pos[1] / self.width)

        for robot_pos in self.robot_positions:
            min_box_dist = min([abs(robot_pos[0] - box_pos[0]) + abs(robot_pos[1] - box_pos[1])
                               for box_pos in self.box_positions
                               if box_pos is not None and
                               not self.delivered_boxes[self.box_positions.index(box_pos)]],
                              default=100)
            obs.append(min_box_dist / (self.height + self.width))

            min_target_dist = min([abs(robot_pos[0] - target_pos[0]) + abs(robot_pos[1] - target_pos[1])
                                  for target_pos in self.targets],
                                 default=100)
            obs.append(min_target_dist / (self.height + self.width))

        return np.array(obs, dtype=np.float32)

    def _is_valid_position(self, pos, robot_id=None):
        """Verifica se a posição é válida"""
        i, j = pos
        if i < 0 or i >= self.height or j < 0 or j >= self.width:
            return False

        cell = self.grid[i][j]
        if cell in ['X', 'Y']:
            return False

        if robot_id is not None:
            for rid, rpos in enumerate(self.robot_positions):
                if rid != robot_id and rpos == (i, j):
                    return False
        return True

    def _move_robot(self, robot_id, action):
        """Move o robô"""
        i, j = self.robot_positions[robot_id]
        new_i, new_j = i, j

        if action == 0: new_i -= 1
        elif action == 1: new_i += 1
        elif action == 2: new_j -= 1
        elif action == 3: new_j += 1

        if self._is_valid_position((new_i, new_j), robot_id):
            old_pos = self.robot_positions[robot_id]
            self.grid[old_pos[0]][old_pos[1]] = '0'
            self.grid[new_i][new_j] = f'R{robot_id + 1}'
            self.robot_positions[robot_id] = (new_i, new_j)
            return True
        else:
            self.collisions += 1
            return False

    def _pickup_box(self, robot_id):
        """Pega uma caixa"""
        robot_pos = self.robot_positions[robot_id]
        for box_id, box_pos in enumerate(self.box_positions):
            if not self.delivered_boxes[box_id] and box_pos == robot_pos:
                self.box_positions[box_id] = None
                return True
        return False

    def _drop_box(self, robot_id):
        """Solta uma caixa no destino"""
        robot_pos = self.robot_positions[robot_id]

        # Encontrar caixa que o robô está carregando
        box_with_robot = None
        for box_id, box_pos in enumerate(self.box_positions):
            if box_pos is None and not self.delivered_boxes[box_id]:
                box_with_robot = box_id
                break

        if box_with_robot is None:
            return False

        # Verificar se está em um destino B
        if robot_pos in self.targets:
            self.delivered_boxes[box_with_robot] = True
            self.total_deliveries += 1
            return True

        return False

    def step(self, actions):
        """Executa um passo no ambiente"""
        self.steps += 1

        # Processar ações dos robôs
        for robot_id, action in enumerate(actions):
            if action < 4:  # Movimento
                self._move_robot(robot_id, action)

        # Processar pegar/soltar
        for robot_id, action in enumerate(actions):
            if action == 4:  # Pegar
                self._pickup_box(robot_id)
            elif action == 5:  # Soltar
                self._drop_box(robot_id)

        terminated = all(self.delivered_boxes)
        truncated = self.steps >= self.max_steps

        return self._get_observation(), terminated, truncated

    def get_state_for_render(self):
        """Retorna o estado atual para renderização"""
        return {
            'grid': [row[:] for row in self.grid],
            'steps': self.steps,
            'deliveries': self.total_deliveries,
            'collisions': self.collisions,
            'remaining_boxes': sum(1 for d in self.delivered_boxes if not d)
        }

    def render_frame(self):
        """Renderiza um único frame"""
        fig, ax = plt.subplots(figsize=(10, 8))

        colors = {
            '0': 'white', 'X': 'black', 'Y': 'gray',
            'R1': '#FF4444', 'R2': '#4444FF', 'A': '#FFA500', 'B': '#00AA00'
        }

        for i in range(self.height):
            for j in range(self.width):
                cell = self.grid[i][j]
                color = colors.get(cell, 'white')
                rect = Rectangle((j, self.height - 1 - i), 1, 1,
                               facecolor=color, edgecolor='black', linewidth=0.5)
                ax.add_patch(rect)

                if cell in ['R1', 'R2', 'A', 'B']:
                    ax.text(j + 0.5, self.height - 0.5 - i, cell,
                           ha='center', va='center', fontweight='bold', fontsize=10)

        ax.set_xlim(0, self.width)
        ax.set_ylim(0, self.height)
        ax.set_xticks(range(self.width))
        ax.set_yticks(range(self.height))
        ax.grid(True, alpha=0.3)
        ax.set_title(f'Warehouse | Steps: {self.steps} | Entregas: {self.total_deliveries}/8 | Colisões: {self.collisions}')

        fig.canvas.draw()

        # Converter para array RGB
        try:
            img = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
            img = img.reshape(fig.canvas.get_width_height()[::-1] + (3,))
        except:
            img = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8)
            img = img.reshape(fig.canvas.get_width_height()[::-1] + (4,))[:, :, :3]

        plt.close(fig)
        return img

# ==================== FUNÇÃO PRINCIPAL ====================
def load_and_visualize(models_dir="/content/drive/MyDrive/Atividades/PUC-DI/LocalMultiAgente/Código/warehouse_results_20260605_165855/models/", output_video="robot_movement.mp4", max_steps=500):
    """Carrega os modelos e gera o vídeo"""

    print("=" * 60)
    print("🎬 CARREGANDO MODELOS E GERANDO VÍDEO")
    print("=" * 60)

    # Verificar se os arquivos existem
    model_paths = [
        Path(models_dir) / "agent_0_final.pth",
        Path(models_dir) / "agent_1_final.pth"
    ]

    for path in model_paths:
        if not path.exists():
            print(f"❌ Arquivo não encontrado: {path}")
            print(f"   Certifique-se de que os modelos estão no diretório: {models_dir}")
            return None

    # Criar ambiente
    env = WarehouseEnv()

    # Obter dimensões
    sample_obs = env.reset()
    state_dim = len(sample_obs)
    action_dim = 6  # 0-3: movimentos, 4: pegar, 5: soltar

    print(f"\n📊 Dimensões:")
    print(f"   Estado: {state_dim}")
    print(f"   Ações: {action_dim}")

    # Criar agentes
    agents = [
        IDQNAgent(state_dim, action_dim, 0),
        IDQNAgent(state_dim, action_dim, 1)
    ]

    # Carregar modelos
    print("\n📂 Carregando modelos...")
    for agent, path in zip(agents, model_paths):
        agent.load_model(str(path))

    # Executar um episódio e gravar frames
    print("\n🎥 Gravando vídeo...")
    frames = []
    episode_stats = {'deliveries': 0, 'steps': 0, 'collisions': 0}

    obs = env.reset()
    episode_reward = 0

    for step in range(max_steps):
        # Renderizar frame
        frame = env.render_frame()
        frames.append(frame)

        # Escolher ações (sem exploração)
        actions = [agent.select_action(obs) for agent in agents]

        # Executar ações
        next_obs, terminated, truncated = env.step(actions)
        obs = next_obs

        if terminated or truncated:
            # Capturar frame final
            final_frame = env.render_frame()
            frames.append(final_frame)
            break

    # Obter estatísticas finais
    state = env.get_state_for_render()
    episode_stats = {
        'deliveries': state['deliveries'],
        'steps': state['steps'],
        'collisions': state['collisions'],
        'remaining_boxes': state['remaining_boxes']
    }

    print(f"\n📊 Estatísticas do Episódio:")
    print(f"   • Entregas: {episode_stats['deliveries']}/8")
    print(f"   • Passos: {episode_stats['steps']}")
    print(f"   • Colisões: {episode_stats['collisions']}")
    print(f"   • Caixas restantes: {episode_stats['remaining_boxes']}")

    # Salvar vídeo
    if frames:
        print(f"\n💾 Salvando vídeo com {len(frames)} frames...")

        # Tentar diferentes métodos de salvamento
        try:
            # Método 1: imageio
            imageio.mimsave(output_video, frames, fps=5)
            print(f"✅ Vídeo salvo (imageio): {output_video}")
        except Exception as e:
            print(f"⚠️ Erro com imageio: {e}")

            try:
                # Método 2: matplotlib animation
                fig, ax = plt.subplots(figsize=(10, 8))

                def update_frame(frame_idx):
                    ax.clear()
                    img = frames[frame_idx]
                    ax.imshow(img)
                    ax.axis('off')
                    return ax,

                anim = FuncAnimation(fig, update_frame, frames=len(frames), interval=200, blit=True)
                anim.save(output_video, writer='ffmpeg', fps=5)
                plt.close(fig)
                print(f"✅ Vídeo salvo (matplotlib): {output_video}")
            except Exception as e2:
                print(f"⚠️ Erro com matplotlib: {e2}")

                # Método 3: salvar frames como imagens individuais
                output_dir = Path("frames")
                output_dir.mkdir(exist_ok=True)
                for i, frame in enumerate(frames):
                    from PIL import Image
                    img = Image.fromarray(frame)
                    img.save(output_dir / f"frame_{i:04d}.png")
                print(f"✅ Frames salvos como imagens em: {output_dir}")
                print(f"   Use um programa externo para compilar o vídeo")

    env.close()

    # Criar gráfico de resumo
    create_summary_plot(episode_stats)

    return frames, episode_stats

def create_summary_plot(stats):
    """Cria um gráfico de resumo do desempenho"""
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    # Dados
    metrics = ['Entregas', 'Passos', 'Colisões']
    values = [stats['deliveries'], stats['steps'], stats['collisions']]
    colors = ['green', 'orange', 'red']

    # Gráfico de barras
    axes[0].bar(metrics, values, color=colors, alpha=0.7)
    axes[0].set_title('Desempenho do Episódio')
    axes[0].set_ylabel('Valor')
    axes[0].grid(True, alpha=0.3)

    # Texto com estatísticas
    axes[1].axis('off')
    success_rate = (stats['deliveries'] / 8) * 100
    efficiency = stats['steps'] / max(stats['deliveries'], 1)

    text = f"""
    📊 RESUMO DO EPISÓDIO

    Entregas: {stats['deliveries']}/8
    Taxa de Sucesso: {success_rate:.1f}%
    Passos: {stats['steps']}
    Colisões: {stats['collisions']}
    Eficiência: {efficiency:.1f} passos/entrega

    {'✅ MISSÃO COMPLETA!' if stats['deliveries'] == 8 else '⚠️ MISSÃO INCOMPLETA'}
    """

    axes[1].text(0.1, 0.5, text, fontsize=12, verticalalignment='center')
    axes[1].set_title('Resumo')

    plt.tight_layout()
    plt.savefig("summary_plot.png", dpi=150)
    plt.close()
    print("📊 Gráfico de resumo salvo: summary_plot.png")

# ==================== EXECUÇÃO ====================
if __name__ == "__main__":
    import sys

    # Diretório onde estão os modelos
    # Altere para o caminho correto onde seus arquivos .pth estão salvos
    MODELS_DIR = "/content/drive/MyDrive/Atividades/PUC-DI/LocalMultiAgente/Código/warehouse_results_20260605_165855/models/"  # Diretório atual
    OUTPUT_VIDEO = "/content/drive/MyDrive/Atividades/PUC-DI/LocalMultiAgente/Output/robot_movement.mp4"


    print(f"\n📁 Diretório dos modelos: {MODELS_DIR}")
    print(f"📹 Vídeo de saída: {OUTPUT_VIDEO}")

    # Executar visualização
    frames, stats = load_and_visualize(MODELS_DIR, OUTPUT_VIDEO)

    if frames:
        print(f"\n✨ VISUALIZAÇÃO CONCLUÍDA COM SUCESSO!")
        print(f"📹 Vídeo disponível em: {OUTPUT_VIDEO}")
        print(f"📊 Gráfico disponível em: summary_plot.png")

        # Tentar reproduzir o vídeo no notebook (se estiver no Colab)
        try:
            from IPython.display import Video
            display(Video(OUTPUT_VIDEO))
        except:
            pass
    else:
        print("\n❌ Falha ao gerar visualização. Verifique os arquivos de modelo.")


📁 Diretório dos modelos: /content/drive/MyDrive/Atividades/PUC-DI/LocalMultiAgente/Código/warehouse_results_20260605_165855/models/
📹 Vídeo de saída: /content/drive/MyDrive/Atividades/PUC-DI/LocalMultiAgente/Output/robot_movement.mp4
🎬 CARREGANDO MODELOS E GERANDO VÍDEO

📊 Dimensões:
   Estado: 40
   Ações: 6

📂 Carregando modelos...
✅ Modelo do Agente 0 carregado de: /content/drive/MyDrive/Atividades/PUC-DI/LocalMultiAgente/Código/warehouse_results_20260605_165855/models/agent_0_final.pth
✅ Modelo do Agente 1 carregado de: /content/drive/MyDrive/Atividades/PUC-DI/LocalMultiAgente/Código/warehouse_results_20260605_165855/models/agent_1_final.pth

🎥 Gravando vídeo...

📊 Estatísticas do Episódio:
   • Entregas: 8/8
   • Passos: 23
   • Colisões: 0
   • Caixas restantes: 0

💾 Salvando vídeo com 24 frames...


✅ Vídeo salvo (imageio): /content/drive/MyDrive/Atividades/PUC-DI/LocalMultiAgente/Output/robot_movement.mp4


AttributeError: 'WarehouseEnv' object has no attribute 'close'